# Get Kegg Pathway Nodes
From the kegg pathways selected below filter out relevant edges and nodes, that are also represented in the L1000 database.
Write the available nodes to file in data_phase1 to use for further analysis

In [7]:
# --- Step 1: Download and parse multiple KEGG pathways ---

# make a pathway id dictionary with pathway_name as key and pathway_id as value
pathway_id_dict = {
    "MAPK_signaling": "hsa04010",
    "T_cell_receptor_signaling": "hsa04660",
    "TGF-beta_signaling": "hsa04350",
    "p53_signaling": "hsa04115",
    "Pathways_in_cancer": "hsa05200",
    "mtor_signaling": "hsa04150",
    "pi3k_akt_signaling": "hsa04151",
    "apoptosis": "hsa04210",
    "tnf_signaling": "hsa04668",
    "nf_kb_signaling": "hsa04064",
    "breast_cancer": "hsa05224",
    "colorectal_cancer": "hsa05210",
    "non_small_cell_lung_cancer": "hsa05223",
    "small_cell_lung_cancer": "hsa05222",
    "transcriptional_misregulation_in_cancer": "hsa05202",
    "central_carbon_metabolism_in_cancer": "hsa05230",
    "cell_cycle": "hsa04110",
    "EGFR_signaling": "hsa01521"
}

In [8]:
from Bio.KEGG.KGML import KGML_parser
from Bio.KEGG import REST
from io import StringIO
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import os

In [9]:
def get_geneid_mapping():
    """
    Fetch gene symbols for a list of KEGG gene IDs.
    """
    df_gene_info = pd.read_csv('data_phase1/gene_info.txt', sep='\t')  
    # create a map by using the columns 'gene_id' and 'gene_symbol'
    L1000_gene_map = df_gene_info.set_index('pr_gene_id')['pr_gene_symbol'].to_dict()
    # to every gene_id in the mapping add "hsa:" as a prefix
    L1000_gene_map = {f"hsa:{k}": v for k, v in L1000_gene_map.items()}
    return L1000_gene_map

In [10]:
def download_pathway(pathway_id):
    kgml = REST.kegg_get(pathway_id, "kgml").read()
    return KGML_parser.read(StringIO(kgml))

In [11]:
pathways = pathway_id_dict.keys()
NETWORK_NAME = "TP53"
base_node= "TP53" #node that should define the subgraph, if none largest component is used

pathways = [download_pathway(pathway_id) for pathway_id in [
    pathway_id_dict[pathway] for pathway in pathways
]]



In [12]:
# --- Create Graph from pathways ---

from os import mkdir
import json
from pathlib import Path

gene_id_to_symbol = get_geneid_mapping()
kegg_nodes_path = Path("kegg_data/kegg_nodes.json")
id_to_human = {v: k for k, v in pathway_id_dict.items()}
parent_dir = kegg_nodes_path.parent
# Fail fast if target directory does not exist
if not parent_dir.exists():
    raise FileNotFoundError(f"Required directory does not exist: {parent_dir}. Create it before running this cell.")

# Load existing data if present, otherwise start with empty dict
if kegg_nodes_path.exists():
    with open(kegg_nodes_path, "r") as f:
        data = json.load(f)
    if not isinstance(data, dict):
        raise ValueError(f"Existing file {kegg_nodes_path} does not contain a JSON object (expected dict).")
else:
    data = {}

activation = ["activation", "expression"] # self loops should not be expression
inhibition = ["inhibition", "repression"]
other = ["compound", "hidden compound", "indirect effect", "state change", "binding/association", 
         "dissociation", "missing interaction", "phosphorylation", "dephosphorylation", "glycosylation", 
         "ubiquitination", "methylation"]

for pw in pathways:
    G = nx.DiGraph()
    id_to_gene_ids = {}      # maps KEGG node ID to list of gene IDs (e.g. ['hsa:7157', 'hsa:1029'])

    added_genes = 0
    skipped_entirely = 0

    # Add gene nodes

    for gene in pw.genes:
        gene_ids = gene.name.split()
        for gid in gene_ids:
            if gid not in gene_id_to_symbol:
                continue
            else:
                G.add_node(gid, label=gene_id_to_symbol[gid], type="gene")
                added_genes += 1
            id_to_gene_ids[gene.id] = [gid for gid in gene_ids if gid in gene_id_to_symbol]

    print(f"✅ Added {added_genes} Genes to the graph.")

    # Add edges based on relations

    activating_edges = 0
    inhibiting_edges = 0
    skipped_edges = 0

    for rel in pw.relations:
        src_id = rel.entry1.id
        tgt_id = rel.entry2.id

        # Only add edge if both sides are gene-type nodes
        if src_id in id_to_gene_ids and tgt_id in id_to_gene_ids:
            src_genes = id_to_gene_ids[src_id]
            tgt_genes = id_to_gene_ids[tgt_id]

            for subtype in rel.subtypes:
                interaction = subtype[0]
                # Determine the type of interaction based on the subtype
                if interaction in activation:
                    interaction = "1"
                    activating_edges += len(src_genes) * len(tgt_genes)
                elif interaction in inhibition:
                    interaction = "2"
                    inhibiting_edges += len(src_genes) * len(tgt_genes)
                elif interaction in other:
                    interaction = "3"
                    skipped_edges += len(src_genes) * len(tgt_genes)
                    continue
                else:
                    "⚠️ Unknown interaction type: {interaction} (subtype: {subtype})"

                # Connect each source gene to each target gene
                for src_gene in src_genes:
                    for tgt_gene in tgt_genes:
                        G.add_edge(src_gene, tgt_gene, interaction=interaction)

    
    G = nx.relabel_nodes(G, {node: G.nodes[node]["label"] for node in G.nodes() if "label" in G.nodes[node]})

    print(f"✅ Added {G.number_of_edges()} edges to the graph.")
    print(f"✅ {activating_edges} activating edges, {inhibiting_edges} inhibiting edges, "
        f"{skipped_edges} skipped edges due to unknown interaction types.")
    
    # remove isolated nodes
    G.remove_nodes_from(list(nx.isolates(G)))
    print(f"✅ Removed isolated nodes. Graph now has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

    wcc = list(nx.weakly_connected_components(G))
    print(f"✅ Graph has {len(wcc)} weakly connected components.")
    wcc_sizes = [len(c) for c in wcc]
    print(f"✅ Sizes of weakly connected components: {wcc_sizes}")

    # choose largest weakly connected component
    largest_wcc = max(wcc, key=len)
    G = G.subgraph(largest_wcc).copy()
    print(f"✅ Selected largest weakly connected component. Graph now has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")


    # --- Step 2: Create or update KEGG pathway -> gene symbol mapping file ---
    raw = getattr(pw, "name", None) or getattr(pw, "title", None)
    code = raw.split(":", 1)[1] if ":" in raw else raw

    if code in id_to_human:
        human_key = id_to_human[code].replace("_", " ").title()
        key = f"{human_key} ({code})"
    else:
        # if mapping missing, use the pathway title if available, otherwise fail fast
        title = getattr(pw, "title", None)
        if title:
            key = f"{title} ({code})"
        else:
            raise ValueError(f"Pathway id {code!r} not found in pathway_id_dict and pathway has no title to build a readable key.")

    if key in data:
        # already recorded -> skip
        continue

    # Use the already-constructed graph `G` to get the node list for this pathway.
    # After earlier processing we have relabelled nodes to gene symbols and removed isolates,
    # so `list(G.nodes())` gives the unique, ordered gene symbols.
    nodes = list(G.nodes())
    n_edges = len(list(G.edges()))

    if not nodes:
        # Fail fast if nothing mapped for this pathway
        raise ValueError(f"No mappable genes found for pathway '{key}'.")

    # Store as dict with 'nodes' key for future extensibility
    data[key] = {"nodes": nodes, "edges": n_edges}

print(f"✅ KEGG pathway to gene symbol mapping now has {len(data)} entries.")
# write back to file (create or update)
with open(kegg_nodes_path, "w") as f:
    print(f"✅ Writing KEGG pathway to gene symbol mapping to {kegg_nodes_path}")
    json.dump(data, f, indent=2, sort_keys=True)
    



✅ Added 291 Genes to the graph.
✅ Added 1484 edges to the graph.
✅ 1349 activating edges, 135 inhibiting edges, 464 skipped edges due to unknown interaction types.
✅ Removed isolated nodes. Graph now has 178 nodes and 1484 edges.
✅ Graph has 6 weakly connected components.
✅ Sizes of weakly connected components: [25, 27, 42, 74, 8, 2]
✅ Selected largest weakly connected component. Graph now has 74 nodes and 1225 edges.
✅ Added 115 Genes to the graph.
✅ Added 236 edges to the graph.
✅ 167 activating edges, 69 inhibiting edges, 200 skipped edges due to unknown interaction types.
✅ Removed isolated nodes. Graph now has 99 nodes and 236 edges.
✅ Graph has 3 weakly connected components.
✅ Sizes of weakly connected components: [83, 8, 8]
✅ Selected largest weakly connected component. Graph now has 83 nodes and 210 edges.
✅ Added 133 Genes to the graph.
✅ Added 187 edges to the graph.
✅ 112 activating edges, 114 inhibiting edges, 72 skipped edges due to unknown interaction types.
✅ Removed iso